# Positional xG Model

In [3]:
from dotenv import load_dotenv
import os
from supabase import create_client, Client
from pathlib import Path
import pandas as pd
from rich.progress import track, Progress
import time
from rich.console import Console
from tqdm.notebook import tqdm
import football_analytics.data_processing.preprocessing as preprocessing
from football_analytics.utils import parsing, supabase, shot_geometry
import football_analytics.visuals.shots as shots
from sklearn.metrics import log_loss
from importlib import reload
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
reload(preprocessing)
reload(parsing)
reload(supabase)
reload(shot_geometry)
reload(shots)

<module 'football_analytics.visuals.shots' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\visuals\\shots.py'>

### Load Supabase Data

In [37]:
table_name = 'shots'
key_column = 'statsbomb_event_id'

In [3]:
data = supabase.fetch_all_rows_in_batches(table_name=table_name, key_column=key_column, batch_size=5000)

Fetched 5000 rows (total 5000).
Fetched 5000 rows (total 10000).
Fetched 5000 rows (total 15000).
Fetched 5000 rows (total 20000).
Fetched 5000 rows (total 25000).
Fetched 5000 rows (total 30000).
Fetched 5000 rows (total 35000).
Fetched 5000 rows (total 40000).
Fetched 5000 rows (total 45000).
Fetched 5000 rows (total 50000).
Fetched 5000 rows (total 55000).
Fetched 5000 rows (total 60000).
Fetched 5000 rows (total 65000).
Fetched 5000 rows (total 70000).
Fetched 5000 rows (total 75000).
Fetched 5000 rows (total 80000).
Fetched 5000 rows (total 85000).
Fetched 3023 rows (total 88023).


In [38]:
df = pd.DataFrame(data)
# df.info()

## Plot Data

In [39]:
cond1 = df['outcome']=='Goal'
cond2 = df['shot_type']!='Penalty'

In [40]:
df_nopens = df[cond2].copy()

In [ ]:
shots.plot_shot_heatmap(df_nopens[['x1', 'y1', 'outcome', 'shot_type', 'shot_taker']])

In [ ]:
reload(shots)
shots.plot_shot_conversion_heatmap(df_nopens[['x1', 'y1', 'outcome']], min_shots=20, plot_3d=True)

In [9]:
print(f"Total shot conversion rate: {100 * (df_nopens['outcome'] == 'Goal').sum() / len(df_nopens):.2f}%")

Total shot conversion rate: 10.15%


## Prepare Data

In [43]:
cond1 = df['shot_type']!='Penalty'
df_nopens = df[cond1].copy()

In [91]:
df['goal_outcome'] = df['outcome'].apply(lambda x: 1 if x=='Goal' else 0)
df['is_with_feet'] = df['body_part'].apply(lambda x: 1 if (x=='Right Foot' or x=='Left Foot') else 0)
df['is_penalty'] = df['shot_type'].apply(lambda x: 1 if x=='Penalty' else 0)
df['avg_goal'] = df['shot_type'].apply(lambda x: 0.78 if x=='Penalty' else 0.1015)
df['all_goal'] = 1
df['no_goal'] = 0

## xG Model

#### Loss References

In [92]:
def xg_log_loss(y_true, y_pred):
    """
    Computes binary cross-entropy (log loss) for xG predictions.
    """
    return log_loss(y_true, y_pred)


In [94]:
xg_log_loss_baseline = xg_log_loss(df['goal_outcome'], df['avg_goal'])
print(f"Baseline log loss (avg goal by shot type): {xg_log_loss_baseline:.4f}")
xg_log_loss_worst = xg_log_loss(df['goal_outcome'], df['no_goal'])
print(f"Baseline log loss (worst case): {xg_log_loss_worst:.4f}")

Baseline log loss (avg goal by shot type): 0.3322
Baseline log loss (worst case): 4.0088


In [95]:
xg_log_loss_statsbomb = xg_log_loss(df['goal_outcome'], df['statsbomb_xg'])
print(f"StatsBomb xG log loss: {xg_log_loss_statsbomb:.4f}")

StatsBomb xG log loss: 0.2673


#### xG Model

In [96]:
# x,y in meters relative to goal center
x_shot = 120 - df['x1'].values
y_shot = 40 - df['y1'].values

r = np.hypot(x_shot, y_shot)
theta = np.abs(np.arctan2(y_shot, x_shot))

# Full dataset
X = np.column_stack([r, theta, df['is_with_feet'].values, df['is_penalty'].values])
y = df['goal_outcome'].values.astype(int)

In [97]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

clf = LogisticRegression(l1_ratio=0.0, C=1.0)
clf.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [98]:
xg_pred = clf.predict_proba(X_test)[:, 1]

In [99]:
xg_log_loss_positional = xg_log_loss(y_test, xg_pred)
print(f"Positional xG log loss: {xg_log_loss_positional:.4f}")

Positional xG log loss: 0.2952
